# Compiling SdOB stars 
## from Joris Vos

based on 
### "The orbital period-mass ratio relation of wide sdB+MS binaries and its application to the stability of RLOF  "
Vos 2019
 https://ui.adsabs.harvard.edu/abs/2019MNRAS.482.4592V/abstract
*** 

Using all wide sdB binaries with known orbital parameters, 23 systems,


In [95]:
import json
from urllib.parse import quote

import numpy as np
import astropy.units as u
from astroquery.vizier import Vizier
import re


# Ensure project root is on sys.path so `import paths` finds the top-level paths.py
import os, sys
from pathlib import Path
proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

from paths import DATA_DIR, RAW_JSON_DIR

In [96]:
table1_path = DATA_DIR / "latex_input_data" / "Vos2019_table1.tex"
table2_path = DATA_DIR / "latex_input_data" / "Vos2019_table2.tex"
table3_path = DATA_DIR / "latex_input_data" / "Vos2019_table3.tex"
table4_path = DATA_DIR / "latex_input_data" / "Vos2019_table4.tex"


In [107]:

# ── helpers ──────────────────────────────────────────────────────────────────

def clean_latex(s):
    """Strip common LaTeX markup and normalise whitespace."""
    s = str(s)
    s = s.replace(r"$^{\circ}$", "°")                     # e.g. BD+34$^{\circ}$1543 -> BD+34°1543
    s = s.replace(r"\,", " ")                              # thin space in names: PB\,6355 -> PB 6355
    s = re.sub(r'\\[;!~ ]', ' ', s)                        # other spacing commands
    s = re.sub(r'\$([^$]*)\$', r'\1', s)                  # inline math
    s = re.sub(r'\\circ', '°', s)
    s = re.sub(r'\{([^}]*)\}', r'\1', s)                  # braces
    s = re.sub(r'\\[a-zA-Z]+', '', s)                      # remaining commands
    s = s.replace('--', '-')                                 # LaTeX double dash -> single
    s = re.sub(r'[\u2010\u2011\u2012\u2013\u2014\u2015\u2212]', '-', s)  # unicode dashes -> '-'
    return re.sub(r'\s+', ' ', s).strip()

def normalize_key(s):
    """Canonical key for matching names across tables."""
    s = clean_latex(s)
    s = re.sub(r'^GALEX\s*', '', s, flags=re.IGNORECASE)
    s = re.sub(r'[\u2010\u2011\u2012\u2013\u2014\u2015\u2212]', '-', s)
    s = re.sub(r'[\s\-+°]+', '', s)
    return s.lower()

def hms_to_deg(hms):
    """'HH MM SS.s' → decimal degrees."""
    h, m, s = hms.strip().split()
    return 15.0 * (float(h) + float(m)/60 + float(s)/3600)

def dms_to_deg(dms):
    """'+DD MM SS.s' or '-DD MM SS.s' → decimal degrees."""
    dms = dms.strip()
    sign = -1 if dms[0] == '-' else 1
    d, m, s = dms.lstrip('+-').split()
    return sign * (float(d) + float(m)/60 + float(s)/3600)

def expand_multicolumn(text):
    """Replace \\multicolumn{n}{fmt}{val} with n '&'-separated copies of val."""
    def repl(m):
        n, val = int(m.group(1)), m.group(3).strip()
        return ' & '.join([val] * n)
    return re.sub(r'\\multicolumn\{(\d+)\}\{([^}]*)\}\{([^}]*)\}', repl, text)

def parse_tabular_rows(tex_path):
    """Return data rows from a LaTeX tabular as lists of stripped strings."""
    text = Path(tex_path).read_text()
    text = re.sub(r'(?<!\\)%[^\n]*', '', text)
    text = expand_multicolumn(text)
    body_match = re.search(r'\\hline\s*\\hline(.*?)\\end\{tabular\}', text, re.DOTALL)
    body = body_match.group(1) if body_match else text
    rows = []
    for line in body.split('\\\\'):
        line = line.strip()
        line = re.sub(r'^\s*\\hline\s*', '', line)
        if not line:
            continue
        cols = [c.strip() for c in line.split('&')]
        if not any(cols):
            continue

        first_col = clean_latex(cols[0]).strip().lower()
        if first_col == 'object' or first_col == '':
            continue

        rows.append(cols)
    return rows


In [108]:
# ── Table 1: coordinates + classification (11 new systems) ───────────────────
t1 = {}
for cols in parse_tabular_rows(table1_path):
    if len(cols) < 5:
        continue
    raw_name, cls, vmag, ra_hms, dec_dms = cols[0], cols[1], cols[2], cols[3], cols[4]
    name = clean_latex(raw_name)
    key  = normalize_key(raw_name)
    # RA is in hours ('HH MM SS.s'), Dec in degrees '+DD MM SS.s'
    try:
        ra_deg  = hms_to_deg(ra_hms)
        dec_deg = dms_to_deg(dec_dms)
    except Exception:
        ra_deg = dec_deg = None
    # Classification: 'sdB+F/G' → obs_type_1='sdB', obs_type_2='F/G'
    cls_clean = clean_latex(cls)
    parts = [p.strip() for p in cls_clean.split('+', 1)]
    obs2 = parts[0] if parts else None # 2 is donor is sdB
    obs1 = parts[1] if len(parts) > 1 else None
    t1[key] = dict(name=name, ra=ra_deg, dec=dec_deg, obs_type_1=obs1, obs_type_2=obs2)


In [109]:
print(f"Table 1: {len(t1)} systems")
for k, v in t1.items():
    print(
        f"  {k:35s}  "
        f"name={str(v.get('name')):25s}  "
        f"ra={v.get('ra')}  "
        f"dec={v.get('dec')}  "
        f"obs_type_1={v.get('obs_type_1')}  "
        f"obs_type_2={v.get('obs_type_2')}"
    )


Table 1: 11 systems
  pb6355                               name=PB 6355                    ra=19.11375  dec=6.053222222222222  obs_type_1=F  obs_type_2=sdB
  mct01462651                          name=MCT 0146-2651              ra=27.183333333333334  dec=-26.603555555555555  obs_type_1=F/G  obs_type_2=sdB
  faust321                             name=FAUST 321                  ra=27.8475  dec=-75.81080555555555  obs_type_1=F  obs_type_2=sdB
  jl277                                name=JL 277                     ra=30.39333333333333  dec=-53.728750000000005  obs_type_1=F5  obs_type_2=sdB
  j022836.7362543                      name=GALEX J022836.7-362543     ra=37.15375  dec=-36.429361111111106  obs_type_1=K0  obs_type_2=sdB
  ec031435945                          name=EC 03143-5945              ra=48.875416666666666  dec=-59.56802777777778  obs_type_1=F9  obs_type_2=sdB
  j033216.7023302                      name=GALEX J033216.7-023302     ra=53.069583333333334  dec=-2.5505277777777775  obs_

In [110]:

# ── Table 2: orbital parameters (11 new systems) ─────────────────────────────
# tabular format r@{±}l per pair, so data cols after Object are:
#   [1] P  [2] P_err  [3] T0  [4] T0_err  [5] e  [6] e_err
#   [7] omega  [8] omega_err  [9] K_MS  [10] K_MS_err  ...
# J053939.1: \multicolumn{2}{c}{0} → "0 & 0" for e; \multicolumn{2}{c}{/} → "/ & /" for omega

def safe_float(s):
    try:
        return float(s)
    except (ValueError, TypeError):
        return None

t2 = {}
for cols in parse_tabular_rows(table2_path):
    if len(cols) < 6:
        continue
    key   = normalize_key(cols[0])
    p_val = safe_float(cols[1])
    p_err = safe_float(cols[2])
    e_val = safe_float(cols[5])
    e_err = safe_float(cols[6])
    t2[key] = dict(period=p_val, period_err=p_err, ecc=e_val, ecc_err=e_err)

print(f"Table 2: {len(t2)} systems")
for k, v in t2.items():
    print(f"  {k:35s}  P={v['period']} ± {v['period_err']}  e={v['ecc']} ± {v['ecc_err']}")


Table 2: 11 systems
  pb6355                               P=684.0 ± 31.0  e=0.22 ± 0.06
  mct01462651                          P=768.0 ± 11.0  e=0.08 ± 0.06
  faust321                             P=993.0 ± 15.0  e=0.1 ± 0.03
  jl277                                P=1082.0 ± 9.0  e=0.15 ± 0.04
  j022836.7362543                      P=554.0 ± 1.0  e=0.15 ± 0.02
  ec031435945                          P=1037.0 ± 3.0  e=0.06 ± 0.02
  j033216.7023302                      P=1247.0 ± 30.0  e=0.18 ± 0.05
  j053939.1283329                      P=865.0 ± 6.0  e=0.0 ± 0.0
  pg1514034                            P=479.0 ± 2.0  e=0.1 ± 0.02
  j162842.0111838                      P=1176.0 ± 30.0  e=0.15 ± 0.05
  pg2148095                            P=1404.0 ± 92.0  e=0.21 ± 0.06


In [111]:
# ── Table 3: P, a, q for all 23 systems ──────────────────────────────────────
# cols: Object P err_P a err_a q err_q
t3 = {}
for cols in parse_tabular_rows(table3_path):
    if len(cols) < 3:
        continue
    key = normalize_key(cols[0])
    name = clean_latex(cols[0])
    def safe_float(s):
        try: return float(s)
        except: return None
    p_val,  p_err  = safe_float(cols[1]), safe_float(cols[2])
    q_val,  q_err  = safe_float(cols[5]), safe_float(cols[6])
    t3[key] = dict(name=name, period=p_val, period_err=p_err, q=q_val, q_err=q_err)

print(f"Table 3: {len(t3)} systems")
for k, v in t3.items():
    print(f"  {k:35s}  P={v['period']} ± {v['period_err']}  q={v['q']} ± {v['q_err']}")


Table 3: 23 systems
  pg1514034                            P=479.0 ± 2.0  q=0.58 ± 0.03
  j022836.7362543                      P=554.0 ± 10.0  q=0.5 ± 0.08
  pb6355                               P=684.0 ± 31.0  q=0.32 ± 0.02
  pg1701359                            P=734.0 ± 15.0  q=None ± None
  pg1018047                            P=752.0 ± 2.0  q=0.7 ± 0.02
  pg1104243                            P=755.0 ± 3.0  q=0.71 ± 0.02
  mct01462651                          P=768.0 ± 11.0  q=0.66 ± 0.03
  ec201174014                          P=795.0 ± 1.0  q=None ± None
  j053939.1283329                      P=865.0 ± 20.0  q=0.74 ± 0.09
  pg1449653                            P=909.0 ± 2.0  q=0.73 ± 0.1
  feige87                              P=938.0 ± 2.0  q=0.55 ± 0.01
  bd341543                             P=972.0 ± 2.0  q=0.57 ± 0.01
  faust321                             P=993.0 ± 15.0  q=0.45 ± 0.01
  ec031435945                          P=1037.0 ± 10.0  q=0.41 ± 0.02
  jl277                

In [112]:
# ── Table 4: sdB masses for all 23 systems ────────────────────────────────────
# cols: Object M_sdB M_sdB_min M_sdB_max

t4 = {}
for cols in parse_tabular_rows(table4_path):
    if len(cols) < 4:
        continue
    key = normalize_key(cols[0])
    def safe_float(s):
        try: return float(s)
        except: return None
    m_cen = safe_float(cols[1])
    m_min = safe_float(cols[2])
    m_max = safe_float(cols[3])
    t4[key] = dict(m_sdB=m_cen, m_sdB_min=m_min, m_sdB_max=m_max)


print(f"Table 4: {len(t4)} systems")
for k, v in t4.items():
    print(f"  {k:35s}  M_sdB={v['m_sdB']}  M_sdB_min={v['m_sdB_min']}  M_sdB_max={v['m_sdB_max']}")


Table 4: 23 systems
  pg1514034                            M_sdB=0.4038  M_sdB_min=0.4035  M_sdB_max=0.404
  j022836.7362543                      M_sdB=0.4135  M_sdB_min=0.4123  M_sdB_max=0.4147
  pb6355                               M_sdB=0.4289  M_sdB_min=0.4255  M_sdB_max=0.4323
  pg1701359                            M_sdB=0.4344  M_sdB_min=0.4328  M_sdB_max=0.436
  pg1018047                            M_sdB=0.4362  M_sdB_min=0.4361  M_sdB_max=0.4365
  pg1104243                            M_sdB=0.4367  M_sdB_min=0.4362  M_sdB_max=0.437
  mct01462651                          M_sdB=0.4379  M_sdB_min=0.437  M_sdB_max=0.4392
  ec201174014                          M_sdB=0.4409  M_sdB_min=0.4409  M_sdB_max=0.441
  j053939.1283329                      M_sdB=0.4482  M_sdB_min=0.4471  M_sdB_max=0.4492
  pg1449653                            M_sdB=0.4526  M_sdB_min=0.4523  M_sdB_max=0.4528
  feige87                              M_sdB=0.4555  M_sdB_min=0.4553  M_sdB_max=0.4558
  bd341543       

In [113]:
print(f"\nKeys in t3 not matched in t4: { set(t3) - set(t4) }")
print(f"Keys in t3 not matched in t1: { set(t3) - set(t1) }")



Keys in t3 not matched in t4: set()
Keys in t3 not matched in t1: {'pg1018047', 'pg1701359', 'feige87', 'ec110311348', 'bd75977', 'bd293070', 'tyc38718351', 'ec201174014', 'pg1104243', 'bd341543', 'pg1449653', 'tyc20844481'}


# Pour data into json table

In [115]:
def triplet(err_lo, val, err_hi):
    """Return [err_lo, val, err_hi] for schema triplets."""
    return [err_lo, val, err_hi]

def value_triplet(val):
    """Return [None, value, None] (or all None) for scalar schema fields."""
    if val is None:
        return [None, None, None]
    return [None, val, None]

def simbad_url(ra, dec, radius_arcmin=2):
    if ra is None or dec is None:
        return None
    return (
        f"https://simbad.u-strasbg.fr/simbad/sim-coo?"
        f"Coord={ra:+.6f}+{dec:+.6f}&CooFrame=FK5&CooEpoch=2000&CooEqui=2000"
        f"&CooDefinedFrames=none&Radius={radius_arcmin}&Radius.unit=arcmin"
        f"&submit=submit+query&CoordList="
    )

entries = []
REFERENCE = "2019MNRAS.482.4592V"

for key, row in t3.items():
    name = row["name"]

    # Period
    p_val = row.get("period")
    p_err = row.get("period_err")
    period = triplet(p_err, p_val, p_err) if p_val is not None else triplet(None, None, None)

    # Eccentricity: prefer t2 (measured), otherwise None
    ecc_rec = t2.get(key, {})
    e_val = ecc_rec.get("ecc")
    e_err = ecc_rec.get("ecc_err")
    eccentricity = triplet(e_err, e_val, e_err) if e_val is not None else triplet(None, None, None)

    # M2 (sdB): from t4
    m_rec = t4.get(key, {})
    m_cen = m_rec.get("m_sdB")
    m_min = m_rec.get("m_sdB_min")
    m_max = m_rec.get("m_sdB_max")
    if m_cen is not None and m_min is not None and m_max is not None:
        M2 = [round(m_cen - m_min, 4), m_cen, round(m_max - m_cen, 4)]
    elif m_cen is not None:
        M2 = [None, m_cen, None]
    else:
        M2 = [None, None, None]

    # M1 (companion): M1 / q  (q = M_sdB / M_MS → M_MS = M_sdB / q)
    q_val = row.get("q")
    if m_cen is not None and q_val is not None and q_val > 0:
        m1_cen = round(m_cen / q_val, 4)
        q_err = row.get("q_err")
        if m_min is not None and q_err is not None:
            frac_M2 = (m_cen - m_min) / m_cen
            frac_q = q_err / q_val
            dm1 = round(m1_cen * (frac_M2**2 + frac_q**2)**0.5, 4)
            M1 = [dm1, m1_cen, dm1]
        else:
            M1 = [None, m1_cen, None]
    else:
        M1 = [None, None, None]

    # Coordinates from t1
    t1_rec = t1.get(key, {})
    ra_deg = t1_rec.get("ra")
    dec_deg = t1_rec.get("dec")
    ra_trip = value_triplet(round(ra_deg, 6) if ra_deg is not None else None)
    dec_trip = value_triplet(round(dec_deg, 6) if dec_deg is not None else None)

    # sdB should always be component 2 (i.e. is past donor)
    obs_type_1 = t1_rec.get("obs_type_1")
    obs_type_2 = t1_rec.get("obs_type_2") or "sdB"

    entry = {
        "System Name": name,
        "RA": ra_trip,
        "Dec": dec_trip,
        "Period": period,
        "Eccentricity": eccentricity,
        "M1": M1,
        "M2": M2,
        "Mass Function": [None, None, None],
        "M1_sin3i": [None, None, None],
        "M2_sin3i": [None, None, None],
        "evol_type_1": "MS",
        "evol_type_2": "He-star",
        "obs_type_1": obs_type_1,
        "obs_type_2": obs_type_2,
        "system_class": "low-M stripped star",
        "Detection Method": ["RV"],
        "Reference": [REFERENCE],
        "Notes": "Hot SdOB star",
        "Simbad": simbad_url(ra_deg, dec_deg),
    }
    entries.append(entry)

print(f"Built {len(entries)} entries")
for e in entries:
    print(f"  {e['System Name']:30s}  P={e['Period']}  e={e['Eccentricity']}  M1={e['M1']}  M2={e['M2']}  RA={e['RA'][1]} Dec={e['Dec'][1]}  obs_types 1,2 ={e['obs_type_1']}, {e['obs_type_2']}")


Built 23 entries
  PG 1514+034                     P=[2.0, 479.0, 2.0]  e=[0.02, 0.1, 0.02]  M1=[0.036, 0.6962, 0.036]  M2=[0.0003, 0.4038, 0.0002]  RA=229.309583 Dec=3.174417  obs_types 1,2 =G6, sdOB
  J022836.7-362543                P=[10.0, 554.0, 10.0]  e=[0.02, 0.15, 0.02]  M1=[0.1323, 0.827, 0.1323]  M2=[0.0012, 0.4135, 0.0012]  RA=37.15375 Dec=-36.429361  obs_types 1,2 =K0, sdB
  PB 6355                         P=[31.0, 684.0, 31.0]  e=[0.06, 0.22, 0.06]  M1=[0.0844, 1.3403, 0.0844]  M2=[0.0034, 0.4289, 0.0034]  RA=19.11375 Dec=6.053222  obs_types 1,2 =F, sdB
  PG 1701+359                     P=[15.0, 734.0, 15.0]  e=[None, None, None]  M1=[None, None, None]  M2=[0.0016, 0.4344, 0.0016]  RA=None Dec=None  obs_types 1,2 =None, sdB
  PG 1018-047                     P=[2.0, 752.0, 2.0]  e=[None, None, None]  M1=[0.0178, 0.6231, 0.0178]  M2=[0.0001, 0.4362, 0.0003]  RA=None Dec=None  obs_types 1,2 =None, sdB
  PG 1104+243                     P=[3.0, 755.0, 3.0]  e=[None, None, None]

## Add missing RA and DEC

In [117]:
# Find systems with missing RA/Dec
missing_coords = []
for e in entries:
    if e['RA'][1] is None or e['Dec'][1] is None:
        missing_coords.append(e['System Name'])

print(f"Systems missing RA/Dec: {len(missing_coords)}")
for name in missing_coords:
    print(f"  - {name}")

Systems missing RA/Dec: 12
  - PG 1701+359
  - PG 1018-047
  - PG 1104+243
  - EC 20117-4014
  - PG 1449+653
  - Feige 87
  - BD+34°1543
  - TYC 2084-448-1
  - EC 11031-1348
  - BD+29°3070
  - BD-7°5977
  - TYC 3871-835-1


In [123]:
# Query SIMBAD for missing RA/Dec with correct name formats
from astroquery.simbad import Simbad

# Use the exact names as they appear (with spaces, keeping +/- signs)
names_to_query = [
    'PG 1701+359',
    'PG 1018-047', 
    'PG 1104+243',
    'EC 20117-4014',
    'PG 1449+653',
    'Feige 87',
    'BD+34 1543',
    'TYC 2084-448-1',
    'EC 11031-1348',
    'BD+29 3070',
    'BD-7 5977',
    'TYC 3871-835-1'
]

simbad_coords = {}
customSimbad = Simbad()

print("Querying SIMBAD for coordinates:")
for name in names_to_query:
    try:
        result = customSimbad.query_object(name)
        if result and len(result) > 0:
            ra = float(result['ra'][0])
            dec = float(result['dec'][0])
            simbad_coords[name] = {'ra': ra, 'dec': dec}
            print(f"  {name:25s}  ✓ RA={ra:.6f}  Dec={dec:.6f}")
        else:
            print(f"  {name:25s}  ✗ NOT FOUND")
    except Exception as e:
        print(f"  {name:25s}  ERROR: {str(e)[:40]}")

print(f"\nFound coordinates for {len(simbad_coords)}/{len(names_to_query)} systems")


Querying SIMBAD for coordinates:
  PG 1701+359                ✓ RA=255.839409  Dec=35.813673
  PG 1018-047                ✓ RA=155.294100  Dec=-4.938772
  PG 1104+243                ✓ RA=166.859302  Dec=24.053086
  EC 20117-4014              ✓ RA=303.769950  Dec=-40.095586
  PG 1449+653                ✓ RA=222.650311  Dec=65.097781
  Feige 87                   ✓ RA=205.061229  Dec=60.879880
  BD+34 1543                 ✓ RA=107.532100  Dec=34.414947
  TYC 2084-448-1             ✓ RA=264.213267  Dec=28.109622
  EC 11031-1348              ✓ RA=166.422566  Dec=-14.073372
  BD+29 3070                 ✓ RA=264.588335  Dec=29.146486
  BD-7 5977                  ✓ RA=349.444950  Dec=-6.475262
  TYC 3871-835-1             ✓ RA=228.909777  Dec=56.895485

Found coordinates for 12/12 systems


In [127]:
# Inject SIMBAD coordinates into entries
# Handle name variations (degree symbol vs space)
name_mapping = {
    'BD+34 1543': 'BD+34°1543',
    'BD+29 3070': 'BD+29°3070',
    'BD-7 5977': 'BD-7°5977',
}

print("Injecting coordinates into entries...")
updated_count = 0
for entry in entries:
    sys_name = entry['System Name']
    query_name = sys_name
    
    # Check if we need to map the name
    for simbad_name, entry_name in name_mapping.items():
        if sys_name == entry_name:
            query_name = simbad_name
            break
    
    if query_name in simbad_coords:
        coords = simbad_coords[query_name]
        ra_deg = coords['ra']
        dec_deg = coords['dec']
        entry['RA'] = value_triplet(round(ra_deg, 6))
        entry['Dec'] = value_triplet(round(dec_deg, 6))
        entry['Simbad'] = simbad_url(ra_deg, dec_deg)
        updated_count += 1
        print(f"  {sys_name:30s}  RA={ra_deg:.6f}  Dec={dec_deg:.6f}")

print(f"\nUpdated {updated_count} entries with SIMBAD coordinates")

# Verify no more missing coordinates
still_missing = [e['System Name'] for e in entries if e['RA'][1] is None or e['Dec'][1] is None]
if still_missing:
    print(f"\nStill missing RA/Dec: {still_missing}")
else:
    print("\n✓ All 23 systems now have RA/Dec coordinates!")


Injecting coordinates into entries...
  PG 1701+359                     RA=255.839409  Dec=35.813673
  PG 1018-047                     RA=155.294100  Dec=-4.938772
  PG 1104+243                     RA=166.859302  Dec=24.053086
  EC 20117-4014                   RA=303.769950  Dec=-40.095586
  PG 1449+653                     RA=222.650311  Dec=65.097781
  Feige 87                        RA=205.061229  Dec=60.879880
  BD+34°1543                      RA=107.532100  Dec=34.414947
  TYC 2084-448-1                  RA=264.213267  Dec=28.109622
  EC 11031-1348                   RA=166.422566  Dec=-14.073372
  BD+29°3070                      RA=264.588335  Dec=29.146486
  BD-7°5977                       RA=349.444950  Dec=-6.475262
  TYC 3871-835-1                  RA=228.909777  Dec=56.895485

Updated 12 entries with SIMBAD coordinates

✓ All 23 systems now have RA/Dec coordinates!


# save the output

In [ ]:
import json

out_path = RAW_JSON_DIR / "Vos2019_sdOB.raw.json"

# with open(out_path, "w") as fh:
#     json.dump(entries, fh, indent=4)


with open(out_path, "w") as f:
    f.write("[\n")
    for i, system in enumerate(entries):
        line = json.dumps(system, separators=(",", ": "), ensure_ascii=False)
        f.write("  " + line)
        if i < len(entries) - 1:
            f.write(",\n")
        else:
            f.write("\n")
    f.write("]\n")

print(f"Wrote {len(entries)} systems to {out_path}")


Wrote 23 systems to /Users/liekevanson/Documents/Projects/post_mt_review/data/result_tables/raw_json/Vos2019_sdOB.raw.json
